# Daily Series Analysis

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [2]:
import mlflow
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from eda_utils import line_with_slider, plot_with_residuals
from plotly.subplots import make_subplots
from statsmodels.tsa.holtwinters import ExponentialSmoothing as HoltWinters

from solar_predictor.data import drop_incomplete_tail, read_sarah3
from solar_predictor.metrics import calculate_metrics
from solar_predictor.storage import get_location_prefix
from solar_predictor.utils import LOCATIONS

px.defaults.height = 700

/home/mario/workspace/solar-predictor/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Setup mlflow
experiment_name = "daily_series_analysis"

mlflow.set_tracking_uri(f"sqlite:///{Path.cwd().parent / 'mlflow.db'}")
exp = mlflow.get_experiment_by_name(experiment_name)
if exp is None:
    mlflow.create_experiment(
        experiment_name, artifact_location=str(Path.cwd().parent / "mlruns")
    )

mlflow.set_experiment(experiment_name)

# Load data

# skopje_lp = get_location_prefix(LOCATIONS["SKOPJE"]["LAT"], LOCATIONS["SKOPJE"]["LONG"])
# raw_df = read_sarah3(skopje_lp)
# raw_df.to_csv("raw_df.csv")

raw_df = pd.read_csv("raw_df.csv", parse_dates=["datetime"]).set_index("datetime")

In [4]:
df = drop_incomplete_tail(raw_df)
daily_df = df.resample("D").mean()

daily_df.tail()

,timestamp,shortwave_radiation,direct_radiation,diffuse_radiation
datetime,,,,
2026-10-26 00:00:00+00:00,1.793015e+09,48.625000,7.812500,40.812500
2026-10-27 00:00:00+00:00,1.793102e+09,49.979167,3.041667,46.937500
2026-10-28 00:00:00+00:00,1.793188e+09,44.375000,2.333333,42.041667
2026-10-29 00:00:00+00:00,1.793274e+09,61.770833,6.604167,55.166667
2026-10-30 00:00:00+00:00,1.793361e+09,49.958333,5.208333,44.750000


# Smoothing

## Simple Moving Average (SMA) and Exponential Moving Average (EMA)

In [5]:
windows = (5, 10, 30)

ma_df = daily_df[["shortwave_radiation"]].copy()
ma_df["prev"] = ma_df["shortwave_radiation"].shift(1)
metrics = []

for w in windows:
    ma_df[f"sma_{w}"] = ma_df["prev"].rolling(w).mean()
    ma_df[f"ema_{w}"] = ma_df["prev"].ewm(span=w, adjust=False, min_periods=w).mean()

    for algorithm in ("sma", "ema"):
        params = {"algorithm": algorithm, "window": w}
        column = f"{algorithm}_{w}"

        with mlflow.start_run(run_name=column):
            run_metrics = calculate_metrics(ma_df["shortwave_radiation"], ma_df[column])
            run_fig = plot_with_residuals(
                ma_df["shortwave_radiation"],
                ma_df[column],
                window=pd.DateOffset(years=1),
                title=f"{algorithm.upper()} (window={w})",
            )
            mlflow.log_params(params)
            mlflow.log_metrics(run_metrics)
            run_fig.write_html("chart.html")
            mlflow.log_figure(figure=run_fig, artifact_file="chart.html")

        metrics.append({**params, **run_metrics})


colors = ["lightgray", *px.colors.qualitative.Plotly]
fig = make_subplots(
    rows=len(windows),
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.02,
)
for row, w in enumerate(windows, start=1):
    columns = {
        "shortwave_radiation": "shortwave_radiation",
        "sma": f"sma_{w}",
        "ema": f"ema_{w}",
    }
    for (name, column), color in zip(columns.items(), colors):
        fig.add_trace(
            go.Scatter(
                x=ma_df.index,
                y=ma_df[column],
                name=name,
                line_color=color,
                legendgroup=name,
                showlegend=row == 1,
            ),
            row=row,
            col=1,
        )
    fig.update_yaxes(title_text=f"{w} days", row=row, col=1)

fig.update_xaxes(range=[pd.to_datetime("2007-01-01"), pd.to_datetime("2008-01-01")])
fig.update_xaxes(
    rangeslider={"visible": True, "thickness": 0.05}, row=len(windows), col=1
)
fig.update_layout(
    title="Shortwave Radiation Moving Averages by Window Size",
    height=800,
    legend={
        "orientation": "h",
        "y": 1.02,
        "x": 1,
        "xanchor": "right",
        "yanchor": "bottom",
    },
)
fig.show()

display(pd.DataFrame(metrics))

,algorithm,window,sse,mse,rmse
0,sma,5,4.823255e+07,3608.330132,60.069378
1,ema,5,4.369299e+07,3254.113995,57.044842
2,sma,10,4.705991e+07,3537.806914,59.479466
3,ema,10,4.394350e+07,3273.990395,57.218794
4,sma,30,5.145506e+07,3934.775290,62.727787
5,ema,30,5.018462e+07,3744.561640,61.192823


Conclusion: Both methods (SMA and EMA) follow the same trend, and larger windows are smoother but react later.
The averages use only previous days, so they don't leak the predicted value.
Model validation error will show which method and window to use.

## Alpha in EMA

In [6]:
ema_df = daily_df[["shortwave_radiation"]].copy()
prev = ema_df["shortwave_radiation"].shift(1)
metrics = []

for alpha in [0.02, 0.08]:
    params = {"algorithm": "ema", "alpha": alpha}
    column = f"ema_{alpha}"

    with mlflow.start_run(run_name=f"ema_alpha_{alpha}"):
        ema_df[column] = prev.ewm(alpha=alpha, adjust=False).mean()
        run_metrics = calculate_metrics(ema_df["shortwave_radiation"], ema_df[column])
        run_fig = plot_with_residuals(
            ema_df["shortwave_radiation"],
            ema_df[column],
            window=pd.DateOffset(years=1),
            title=f"EMA (alpha={alpha})",
        )
        mlflow.log_params(params)
        mlflow.log_metrics(run_metrics)
        run_fig.write_html("chart.html")
        mlflow.log_figure(figure=run_fig, artifact_file="chart.html")

    metrics.append({**params, **run_metrics})


fig = line_with_slider(
    df=ema_df,
    window=pd.DateOffset(years=1),
    color_discrete_map={"shortwave_radiation": "lightgray"},
)
fig.show()

display(pd.DataFrame(metrics))

,algorithm,alpha,sse,mse,rmse
0,ema,0.02,8.562851e+07,6375.437913,79.846339
1,ema,0.08,4.789477e+07,3565.986952,59.715885


## Holt-Winter

### Prepare data
Remove invalid data points and interpolate missing values.

In [7]:
es_df = daily_df[["shortwave_radiation"]].copy()
# Remove zeroes
es_df.loc[es_df["shortwave_radiation"] == 0] = np.nan

es_df["shortwave_radiation"] = es_df["shortwave_radiation"].interpolate(method="time")

### Additive Season

In [8]:
params = {"trend": None, "seasonal_periods": 365, "seasonal": "add"}

with mlflow.start_run(run_name="holtwinters_add"):
    hw_add = HoltWinters(es_df["shortwave_radiation"], **params).fit()
    es_df["holt_winters_add"] = hw_add.fittedvalues

    fig = plot_with_residuals(
        es_df["shortwave_radiation"],
        es_df["holt_winters_add"],
        window=pd.DateOffset(years=1),
        title="Holt-Winters (additive)",
    )
    mlflow.log_params({"algorithm": "holtwinters", **params})
    mlflow.log_metrics(
        {
            **calculate_metrics(
                es_df["shortwave_radiation"], es_df["holt_winters_add"]
            ),
            "aic": hw_add.aic,
        }
    )

    fig.write_html("chart.html")
    mlflow.log_figure(figure=fig, artifact_file="chart.html")

fig.show()

/home/mario/workspace/solar-predictor/.venv/lib/python3.14/site-packages/statsmodels/tsa/holtwinters/model.py:1065: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  res = self._optimize_parameters(res, use_brute, method, minimize_kwargs)


### Multiplicative Season

In [9]:
params = {"trend": None, "seasonal_periods": 365, "seasonal": "mul"}

with mlflow.start_run(run_name="holtwinters_mul"):
    hw_add = HoltWinters(es_df["shortwave_radiation"], **params).fit()
    es_df["holt_winters_add"] = hw_add.fittedvalues

    fig = plot_with_residuals(
        es_df["shortwave_radiation"],
        es_df["holt_winters_add"],
        window=pd.DateOffset(years=1),
        title="Holt-Winters (multiplicative)",
    )
    mlflow.log_params({"algorithm": "holtwinters", **params})
    mlflow.log_metrics(
        {
            **calculate_metrics(
                es_df["shortwave_radiation"], es_df["holt_winters_add"]
            ),
            "aic": hw_add.aic,
        }
    )

    fig.write_html("chart.html")
    mlflow.log_figure(figure=fig, artifact_file="chart.html")

fig.show()

/home/mario/workspace/solar-predictor/.venv/lib/python3.14/site-packages/statsmodels/tsa/holtwinters/model.py:1065: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  res = self._optimize_parameters(res, use_brute, method, minimize_kwargs)
